In [ ]:
%run common_deployment_config

In [ ]:
import json
from pathlib import Path
import fnmatch

# ENFORCE Fabric-only execution
if 'BASE_DIST_PATH' not in globals():
    raise RuntimeError('This notebook must be run in Fabric with common_deployment_config available.')

# Remote manifest path on OneLake/ABFSS
MANIFEST_REMOTE = f"{BASE_DIST_PATH}/healthcare-artifacts-validator-config/build_artifacts_validator_config.json"


def load_manifest_remote(remote_path: str):
    if not notebookutils.fs.exists(remote_path):
        raise FileNotFoundError(f"Remote manifest does not exist: {remote_path}")
    try:
        content = notebookutils.fs.head(remote_path, 10 * 1024 * 1024)
    except Exception as e:
        raise FileNotFoundError(f"Failed to read remote manifest content: {remote_path} ({e})") from e
    try:
        return json.loads(content)
    except json.JSONDecodeError as e:
        raise ValueError(f"Manifest file is not valid JSON: {remote_path} ({e})") from e


def remote_exists(remote_path: str) -> bool:
    try:
        return notebookutils.fs.exists(remote_path)
    except Exception:
        return False


def remote_list(remote_dir: str):
    """Return list of names under remote_dir, or empty list on error."""
    try:
        entries = notebookutils.fs.ls(remote_dir)
        names = []
        for e in entries:
            if hasattr(e, 'name'):
                names.append(e.name)
            elif isinstance(e, dict) and 'name' in e:
                names.append(e['name'])
            else:
                names.append(str(e))
        return names
    except Exception:
        return []


def normalize_manifest(manifest: dict) -> dict:
    """Normalize manifest variants so validator logic can assume dicts for folders.

    Behavior:
    - If a top-level value is a list, convert it to a dict mapping each entry to {}.
    - If a top-level value is a dict and contains '__folder__' (list), convert to dict.
    - If a top-level value is a dict of versions, and a version value is a list or
      an object with '__folder__', convert that version value to a dict.
    This makes the validator tolerant of list / __folder__ shapes across the manifest.
    """
    try:
        for top_key, top_value in list(manifest.items()):
            # Top-level list -> convert to dict {name: {}}
            if isinstance(top_value, list):
                manifest[top_key] = {k: {} for k in top_value}
                continue

            # Top-level dict with '__folder__' -> convert to dict
            if isinstance(top_value, dict) and '__folder__' in top_value and isinstance(top_value['__folder__'], list):
                manifest[top_key] = {k: {} for k in top_value['__folder__']}
                continue

            # Otherwise, treat as dict of versions/children and normalize inner lists/__folder__
            if isinstance(top_value, dict):
                for version_key, version_value in list(top_value.items()):
                    if isinstance(version_value, list):
                        top_value[version_key] = {k: {} for k in version_value}
                        continue
                    if isinstance(version_value, dict) and '__folder__' in version_value and isinstance(version_value['__folder__'], list):
                        top_value[version_key] = {k: {} for k in version_value['__folder__']}
    except Exception:
        pass
    return manifest


def matches_pattern_list(names, pattern):
    pat = pattern.lower()
    return any(fnmatch.fnmatch(n.lower(), pat) for n in names)


def collect_paths(node, base_path=Path()):
    missing = []
    if isinstance(node, list):
        for name in node:
            missing.extend(collect_paths({}, base_path / name))
        return missing
    if not isinstance(node, dict):
        return missing
    for key, value in node.items():
        if key == '__files__' and isinstance(value, list):
            for file_name in value:
                if len(base_path.parts) > 0:
                    remote_parent = f"{BASE_DIST_PATH}/{base_path.as_posix()}"
                else:
                    remote_parent = f"{BASE_DIST_PATH}"
                if '*' in file_name:
                    names = remote_list(remote_parent)
                    if not matches_pattern_list(names, file_name):
                        missing.append(str((base_path / file_name).as_posix()))
                    continue
                if len(base_path.parts) > 0 and base_path.name == '_delta_log' and file_name.lower().endswith('.json'):
                    names = remote_list(remote_parent)
                    if not any(n.lower().endswith('.json') for n in names):
                        missing.append(str((base_path / file_name).as_posix()))
                    continue
                remote_file = f"{BASE_DIST_PATH}/{(base_path / file_name).as_posix()}"
                if not remote_exists(remote_file):
                    missing.append(str((base_path / file_name).as_posix()))
            continue
        if isinstance(value, dict) or isinstance(value, list):
            missing.extend(collect_paths(value, base_path / key))
    return missing


# Load remote manifest
manifest = load_manifest_remote(MANIFEST_REMOTE)
manifest = normalize_manifest(manifest)
print(f"Loaded remote manifest: {MANIFEST_REMOTE}")
print(f"Top-level folders: {len(manifest)}")

# compute missing files for initial summary
missing_files = collect_paths(manifest)

print('Validation Summary:\n')
print(f'  Top-level folders: {len(manifest)}')
print(f'  Missing files: {len(missing_files)}')

if missing_files:
    for i, path in enumerate(missing_files, 1):
        print(f"{i}. {path}")
    raise Exception("Few files are missing, first fix the missing files and re-run the validator.")
else:
    print('\n✅ No missing files found.')